# 01 — JAX Basics

JAX is a numerical computing library that looks like NumPy but adds:
- **Just-in-time compilation** (`jax.jit`) via XLA
- **Automatic differentiation** (`jax.grad`)
- **Vectorisation** (`jax.vmap`)
- **Functional array transformations** (`jax.lax.scan`, etc.)

These four primitives compose cleanly — you can `vmap` a `jit`-ted function, or `grad` through a `scan`. That composability is why JAX is popular for scientific computing and ML research.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

print('JAX version:', jax.__version__)
print('Default backend:', jax.default_backend())
print('Devices:', jax.devices())

## 1.1 Arrays: `jnp` vs `np`

JAX arrays (`jax.Array`) live on an accelerator device. Otherwise the API mirrors NumPy.

In [ ]:
x_np = np.array([1.0, 2.0, 3.0])
x_jax = jnp.array([1.0, 2.0, 3.0])

print('NumPy type:', type(x_np))
print('JAX type:  ', type(x_jax))
print('dtype:     ', x_jax.dtype)
print('device:    ', x_jax.device)

# Convert back when needed
x_back = np.array(x_jax)
print('Back to np:', x_back)

In [ ]:
# JAX defaults to float32 (not float64) to match GPU hardware
print(jnp.array(1.0).dtype)    # float32
print(jnp.array(1.0, dtype=jnp.float64).dtype)  # float64 (if x64 enabled)

# Enable 64-bit globally (useful for scientific computing)
jax.config.update('jax_enable_x64', True)
print('After x64 enable:', jnp.array(1.0).dtype)  # still float32 by default
print(jnp.float64(1.0).dtype)  # float64

## 1.2 JIT Compilation

`jax.jit` traces your function once with abstract values, compiles it to XLA bytecode, then caches the result. Subsequent calls skip Python overhead entirely.

**Key rule**: avoid Python-level conditionals on *array values* inside a JIT-ted function — JAX only sees abstract shapes at trace time, not actual values.

In [ ]:
import time

def slow_fn(x):
    """Matrix multiply, pure NumPy."""
    return np.sin(x) @ np.cos(x).T

@jax.jit
def fast_fn(x):
    """Same op, JIT-compiled."""
    return jnp.sin(x) @ jnp.cos(x).T

x = jnp.ones((500, 500))

# Warm up (first call compiles)
_ = fast_fn(x).block_until_ready()

n = 20
t0 = time.perf_counter()
for _ in range(n): slow_fn(np.array(x))
print(f'NumPy:  {(time.perf_counter()-t0)/n*1000:.2f} ms/call')

t0 = time.perf_counter()
for _ in range(n): fast_fn(x).block_until_ready()
print(f'JAX JIT:{(time.perf_counter()-t0)/n*1000:.2f} ms/call')

In [ ]:
# static_argnums: treat an arg as compile-time constant (re-traces for each value)
@jax.jit
def add_n(x, n):
    return x + n

# This FAILS if n controls array shape or Python branching — use static_argnums instead:
from functools import partial

@partial(jax.jit, static_argnums=(1,))
def repeat_n(x, n):  # n is now a compile-time constant
    return jnp.tile(x, n)

print(repeat_n(jnp.array([1, 2, 3]), 3))

## 1.3 Automatic Differentiation

`jax.grad(f)` returns a function that computes the gradient of `f` with respect to its first argument. `f` must be scalar-valued.

In [ ]:
# Gradient of a scalar function
f = lambda x: jnp.sum(x ** 2)
grad_f = jax.grad(f)

x = jnp.array([1.0, 2.0, 3.0])
print('f(x)      =', f(x))
print('grad f(x) =', grad_f(x))   # Should be 2x = [2, 4, 6]

In [ ]:
# value_and_grad: get both f(x) and ∇f(x) in one forward+backward pass
loss_and_grad = jax.value_and_grad(f)
val, grad = loss_and_grad(x)
print('loss:', val, '  grad:', grad)

# argnums: differentiate w.r.t. a different argument
g = lambda a, b: jnp.dot(a, b)
grad_b = jax.grad(g, argnums=1)(x, x)   # ∂g/∂b = a
print('∂g/∂b =', grad_b)

# Higher-order derivatives (Hessian)
hess_f = jax.jacfwd(jax.grad(f))
print('Hessian of sum(x²):\n', hess_f(x))   # Should be 2*I

## 1.4 Numerical Example: Gradient Descent on a Quadratic

Fit parameters `(a, b, c)` of `p(x) = a x² + b x + c` to noisy observations.

In [ ]:
# Generate noisy data from a known quadratic
key = jax.random.PRNGKey(0)
x_data = jnp.linspace(-3, 3, 50)
true_params = jnp.array([2.0, -1.0, 0.5])   # a=2, b=-1, c=0.5
noise = 0.3 * jax.random.normal(key, x_data.shape)
y_data = true_params[0]*x_data**2 + true_params[1]*x_data + true_params[2] + noise

def predict(params, x):
    a, b, c = params
    return a*x**2 + b*x + c

def loss(params):
    preds = predict(params, x_data)
    return jnp.mean((preds - y_data)**2)

# JIT-compile gradient computation
grad_loss = jax.jit(jax.grad(loss))

params = jnp.zeros(3)   # start at [0, 0, 0]
lr = 0.05
losses = []

for step in range(200):
    g = grad_loss(params)
    params = params - lr * g
    losses.append(float(loss(params)))

print(f'True params:  {np.array(true_params)}')
print(f'Fitted params:{np.array(params)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(losses)
axes[0].set(xlabel='Step', ylabel='MSE loss', title='Training loss')
axes[0].grid(alpha=0.3)

x_plot = jnp.linspace(-3, 3, 200)
axes[1].scatter(x_data, y_data, s=20, label='noisy data', alpha=0.6)
axes[1].plot(x_plot, predict(params, x_plot), 'r-', lw=2, label='fitted')
axes[1].plot(x_plot, predict(true_params, x_plot), 'k--', lw=1.5, label='true')
axes[1].legend()
axes[1].set(title='Fitted quadratic')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 1.5 Random Number Generation in JAX

JAX uses **explicit, functional PRNG** — you always pass a key and split it before reuse. This makes randomness reproducible and parallelisable.

In [ ]:
key = jax.random.PRNGKey(42)

# ALWAYS split before use — never reuse a key
key, subkey1, subkey2 = jax.random.split(key, 3)

x = jax.random.normal(subkey1, (5,))
y = jax.random.uniform(subkey2, (5,))
print('Normal  :', x)
print('Uniform :', y)

# Split N keys at once for batch operations
keys = jax.random.split(key, 8)   # shape (8, 2)
batch_samples = jax.vmap(lambda k: jax.random.normal(k, (3,)))(keys)
print('Batch samples shape:', batch_samples.shape)